In [1]:
import pandas as pd

df = pd.read_csv("datos/originales/dengue_2000_2025.csv",
                 dtype={"ubigeo": str, "localcod": str})   # ubigeo como texto: "010101", no 10101
print(df.shape)
print(df.dtypes)
print(df.head())

print(df["ano"].value_counts().sort_index())            # casos por año: ¿se ven 2017 y 2023?
print(df["tipo_dx"].value_counts(dropna=False))          # C = confirmado, P = probable, S = sospechoso
print(df["enfermedad"].value_counts())                   # sin/con signos de alarma, grave
print(df["semana"].describe())                           # ¿hay semana 53?
print(df.isna().mean().round(3))                         # % de nulos por columna
print("duplicados exactos:", df.duplicated().sum())
print(df["ubigeo"].str.len().value_counts())             # ¿todos con 6 dígitos?

# ── Validación contra la sala CDC ──
cp = df[df["tipo_dx"].isin(["C", "P"])]                  # confirmados + probables
print("2025 año completo (C+P):", (cp["ano"] == 2025).sum(), "| CDC: 35 853")
print("2025 hasta SE 42 (C+P):", ((cp["ano"] == 2025) & (cp["semana"] <= 42)).sum(), "| CDC: 32 594")
print("2023 año completo (C+P):", (cp["ano"] == 2023).sum())

(1065274, 15)
departamento        str
provincia           str
distrito            str
localidad           str
enfermedad          str
ano               int64
semana            int64
diagnostic          str
tipo_dx             str
diresa          float64
ubigeo              str
localcod            str
edad              int64
tipo_edad           str
sexo                str
dtype: object
  departamento      provincia                distrito  \
0      HUANUCO  LEONCIO PRADO                 LUYANDO   
1      HUANUCO  LEONCIO PRADO                 LUYANDO   
2      HUANUCO  LEONCIO PRADO  JOSE CRESPO Y CASTILLO   
3      HUANUCO  LEONCIO PRADO  JOSE CRESPO Y CASTILLO   
4      HUANUCO  LEONCIO PRADO   MARIANO DAMASO BERAUN   

                   localidad                   enfermedad   ano  semana  \
0  SANTA ROSA DE SHAPAJILLLA  DENGUE SIN SIGNOS DE ALARMA  2000      47   
1                 NARANJILLO  DENGUE SIN SIGNOS DE ALARMA  2000      40   
2                   AUCAYACU  DENGUE SIN SIG

In [2]:
# Casos por año y semana, 2023-2024
sem = cp[cp["ano"].isin([2023, 2024])].groupby(["ano", "semana"]).size().unstack(0)
print(sem.idxmax(), sem.max())          # semana pico de cada año
print(sem.iloc[::4])                    # una de cada 4 semanas

# Anomalía Niño 1+2 en esos años
noaa = pd.read_csv("datos/originales/noaa_sstoi_nino12.txt", sep=r"\s+", skiprows=1,
                   names=["anio","mes","t_nino12","anom_nino12","t_nino3","anom_nino3",
                          "t_nino4","anom_nino4","t_nino34","anom_nino34"])
print(noaa[noaa["anio"].isin([2023, 2024])][["anio","mes","anom_nino12"]].to_string(index=False))

ano
2023    20
2024    14
dtype: int64 ano
2023    19269
2024    21628
dtype: int64
ano      2023   2024
semana              
1        1504   1623
5        1759   3818
9        2336  13865
13       3955  17586
17       8385  17155
21      17311   6558
25      10484   1623
29       6342    525
33       3982    440
37       2537    432
41       1605    419
45       1556    824
49       1364   1069
 anio  mes  anom_nino12
 2023    1        -0.24
 2023    2         0.71
 2023    3         1.52
 2023    4         2.45
 2023    5         2.02
 2023    6         2.63
 2023    7         3.21
 2023    8         3.30
 2023    9         2.82
 2023   10         2.46
 2023   11         2.22
 2023   12         1.41
 2024    1         0.83
 2024    2         0.92
 2024    3         0.28
 2024    4         0.08
 2024    5        -0.68
 2024    6        -0.65
 2024    7        -0.41
 2024    8        -0.34
 2024    9        -0.68
 2024   10        -0.32
 2024   11         0.26
 2024   12        -0.10
